# 05 — Accelerated MD and Gaussian accelerated MD

**Learning goals**

- smooth a potential with an energy-dependent boost $\Delta V(V)$ and see how the force changes,
- recover unbiased statistics by reweighting with $e^{\beta\Delta V}$ — and see when that becomes noisy,
- choose GaMD parameters with the standard constraints and examine the boost distribution,
- apply second-order cumulant reweighting and understand what it assumes.

In [ ]:
# Setup: make the `src` package importable (locally or on Google Colab).
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HyeonjeYang/interactive-metadynamics.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/interactive-metadynamics")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / "src").is_dir():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import time as timer
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import skew

from src.potentials import double_well_landmarks, true_free_energy, double_well_potential
from src import accelerated_md as amd
from src.analysis import histogram_free_energy, free_energy_rmse, count_transitions, kish_effective_sample_size
from src.plotting import set_style, save_figure, plot_reference, free_energy_axes, REFERENCE, MUTED, METHOD_COLORS
set_style()

## Parameters

In [ ]:
a, b = 2.0, 0.5
T = 0.4
friction = 1.0
dt = 0.002
n_bath, k_bath = 4, 10.0        # hidden harmonic coordinates, decoupled from x (see Section 1)
n_walkers = 16                  # independent walkers, all started in the left well
n_steps = 60_000                # per walker (120 time units), identical for every method
stride = 10

E_amd, alpha_amd = 2.0, 0.5                 # moderate aMD boost
E_amd_strong, alpha_amd_strong = 5.0, 0.1   # aggressive aMD boost

sigma0_kT = 1.5                 # GaMD upper limit for the boost standard deviation, in kT
gamd_threshold = "lower"        # "lower": E = V_max;  "upper": E = V_min + 1/k (used only if valid)
calib_cmd_steps, calib_rounds, calib_round_steps = 10_000, 3, 10_000

bins = np.linspace(-1.8, 1.7, 71)
min_count = 20                  # frames per bin needed for a reweighted estimate
seed = 2026

## 1. Why a few hidden coordinates?

aMD and GaMD boost the **potential energy** of the whole system, not a chosen coordinate. If the system had only
$x$, then $\Delta V$ would be a fixed function of $x$ and reweighting a histogram in $x$ would be trivially exact.
In molecules, many other degrees of freedom contribute to $V$, so $\Delta V$ fluctuates at fixed $x$. To keep that
feature, $x$ is accompanied by `n_bath` harmonic coordinates that do not interact with it:

$$V(x,\mathbf y) = a(x^2-1)^2 + bx + \sum_m \tfrac12 k_\mathrm{bath}\,y_m^2 .$$

They do not change $F(x)$, so $F_\mathrm{true}(x) = V_\mathrm{dw}(x) - \min V_\mathrm{dw}$ is still the reference.

## 2. The aMD boost

$$\Delta V = \begin{cases}\dfrac{(E-V)^2}{\alpha+E-V}, & V<E\\[4pt] 0, & V\ge E\end{cases}
\qquad V^* = V + \Delta V, \qquad -\nabla V^* = -\frac{\alpha^2}{(\alpha+E-V)^2}\nabla V \;\;(V<E).$$

Every force is scaled by the same factor, which tends to 1 at the threshold, so $V^*$ is smooth there.
The plot shows a slice through the potential with the bath frozen at its average energy $n_\mathrm{bath}k_BT/2$.

In [ ]:
lm = double_well_landmarks(a, b)
system = dict(n_bath=n_bath, k_bath=k_bath, a=a, b=b, friction=friction)
x_plot = np.linspace(-1.7, 1.6, 400)
V_bath = n_bath * T / 2
V_slice = double_well_potential(x_plot, a, b) + V_bath

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(x_plot, V_slice - V_bath, color=REFERENCE, lw=1.4, label=r"$V$")
for (E_, al), ls, name in [((E_amd, alpha_amd), "-", "moderate"), ((E_amd_strong, alpha_amd_strong), "--", "aggressive")]:
    dV = amd.amd_boost(V_slice, E_, al)
    axes[0].plot(x_plot, V_slice + dV - V_bath, ls=ls, color=METHOD_COLORS["amd"], label=rf"$V^*$ {name} (E={E_}, $\alpha$={al})")
    axes[1].plot(x_plot, dV, ls=ls, color=METHOD_COLORS["amd"], label=name)
axes[0].set_ylim(-0.8, 5.5)
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel("energy (bath at mean energy)")
axes[0].legend(fontsize=8)
axes[1].set_xlabel(r"$x$")
axes[1].set_ylabel(r"boost $\Delta V$")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

# Analytical modified force versus a finite difference of V*, in every coordinate
rng_check = np.random.default_rng(0)
q = np.column_stack([rng_check.uniform(-1.5, 1.5, 8), rng_check.normal(0, 0.3, (8, n_bath))])
def v_star(qq):
    V = amd.total_potential(qq, a, b, k_bath)
    return V + amd.amd_boost(V, E_amd, alpha_amd)

V_q = amd.total_potential(q, a, b, k_bath)
analytic = amd.amd_force_scale(V_q, E_amd, alpha_amd)[:, None] * amd.total_gradient(q, a, b, k_bath)
h, eye = 1e-6, np.eye(1 + n_bath)
numeric = np.column_stack([(v_star(q + h * eye[d]) - v_star(q - h * eye[d])) / (2 * h) for d in range(1 + n_bath)])
print(f"max |analytic - numerical| gradient of V*: {np.max(np.abs(analytic - numeric)):.1e}")

The wells are raised; regions above $E$ are untouched. A smaller $\alpha$ and a higher $E$ flatten more of the landscape.

## 3. Ordinary dynamics versus aMD

In [ ]:
x0 = np.full(n_walkers, lm["x_left"])
core = (lm["x_left"] + 0.3, lm["x_right"] - 0.3)
runs = {}
for name, kw in [("ordinary", dict(method="none")),
                 ("aMD (moderate)", dict(method="amd", E=E_amd, alpha=alpha_amd)),
                 ("aMD (aggressive)", dict(method="amd", E=E_amd_strong, alpha=alpha_amd_strong))]:
    t0 = timer.perf_counter()
    runs[name] = amd.run_boosted_dynamics(x0, n_steps, dt, T, np.random.default_rng(seed), stride=stride, **kw, **system)
    r = runs[name]
    print(f"{name:17s} transitions = {count_transitions(r['x'], *core).sum():3d}   "
          f"x range [{r['x'].min():+.2f}, {r['x'].max():+.2f}]   ({timer.perf_counter() - t0:.1f} s)")

fig, axes = plt.subplots(3, 1, figsize=(8, 5), sharex=True, sharey=True)
colors = [METHOD_COLORS["unbiased"], METHOD_COLORS["amd"], METHOD_COLORS["amd"]]
for ax, (name, r), c in zip(axes, runs.items(), colors):
    ax.plot(r["time"], r["x"][:, 0], lw=0.5, color=c)
    ax.set_title(f"{name} (walker 0)", fontsize=9)
    ax.set_ylabel(r"$x$")
axes[-1].set_xlabel("time")
fig.tight_layout()
plt.show()

## 4. Reweighting aMD

A frame with boost $\Delta V$ was over-represented by $e^{-\beta\Delta V}$, so it gets weight
$$w = e^{\beta\Delta V}.$$
The weights are handled in log space. The Kish effective sample size $(\sum w)^2/\sum w^2$ measures how many
frames effectively contribute.

In [ ]:
beta = 1.0 / T
xc = 0.5 * (bins[1:] + bins[:-1])
F_ref = true_free_energy(xc, a, b)
region = F_ref <= lm["barrier_from_left"] + 2 * T

def frames(r, skip=100):
    """Discard the first frames of every walker, then flatten."""
    return r["x"][skip:].ravel(), r["dV"][skip:].ravel()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
plot_reference(axes[1], xc, F_ref)
for name, ls in [("aMD (moderate)", "-"), ("aMD (aggressive)", "--")]:
    x_s, dV_s = frames(runs[name])
    logw = beta * dV_s
    ess = kish_effective_sample_size(logw) / logw.size
    axes[0].hist((logw - logw.max()) / np.log(10), bins=60, weights=np.full(logw.size, 1 / logw.size), log=True,
                 histtype="step", lw=1.6, ls=ls, color=METHOD_COLORS["amd"], label=f"{name}: ESS = {ess:.1%}")
    _, F_rw, _ = histogram_free_energy(x_s, bins, T, log_weights=logw, min_count=min_count)
    err = free_energy_rmse(F_rw, F_ref, region)
    axes[1].plot(xc, F_rw, ls=ls, color=METHOD_COLORS["amd"], label=f"{name}, RMSE {err[0] / T:.2f} kT")
    print(f"{name}: max beta*dV = {logw.max():.1f}, ESS = {ess:.2%} of {logw.size} frames, "
          f"largest single weight = {np.exp(logw.max() - np.logaddexp.reduce(logw)):.2%} of the total")
axes[0].set_xlabel(r"$\log_{10}(w / w_\mathrm{max})$")
axes[0].set_ylabel("fraction of frames")
axes[0].legend(fontsize=8)
free_energy_axes(axes[1], ylim=(-0.2, 4.5))
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

The aggressive boost crosses the barrier far more often, but its weights span several orders of magnitude and
the effective sample size drops by roughly a factor of ten. This toy trajectory is long enough that the reweighted
profile survives. In molecules, where many degrees of freedom contribute to $\Delta V$ and boosts reach tens of
$k_BT$, the exponential average can be dominated by a handful of frames. Exponential reweighting is exact in
principle but can be statistically inefficient; this is the problem GaMD was designed around.

## 5. GaMD: a constrained harmonic boost

$$\Delta V = \tfrac12 k\,(E-V)^2 \;\; (V<E), \qquad -\nabla V^* = -\left[1 - k(E-V)\right]\nabla V.$$

Following Miao et al. (2015), with $V_\mathrm{max}, V_\mathrm{min}, V_\mathrm{avg}, \sigma_V$ from a calibration run:

- monotonic and smoother modified potential: $V_\mathrm{max} \le E \le V_\mathrm{min} + 1/k$, with $k = k_0/(V_\mathrm{max}-V_\mathrm{min})$, $0<k_0\le1$;
- narrow boost: $\sigma_{\Delta V} \approx k(E-V_\mathrm{avg})\sigma_V \le \sigma_0$;
- lower bound $E=V_\mathrm{max}$: $k_0 = \min\!\left(1, \frac{\sigma_0}{\sigma_V}\frac{V_\mathrm{max}-V_\mathrm{min}}{V_\mathrm{max}-V_\mathrm{avg}}\right)$;
- upper bound $E = V_\mathrm{min}+1/k$: $k_0 = \left(1-\frac{\sigma_0}{\sigma_V}\right)\frac{V_\mathrm{max}-V_\mathrm{min}}{V_\mathrm{avg}-V_\mathrm{min}}$ if that lies in $(0,1]$, otherwise the lower-bound rule.

The calibration below is a simplified version of the standard procedure: conventional dynamics, then a few short
boosted rounds that update the statistics. The parameters are then **frozen** for production.

In [ ]:
params, history = amd.calibrate_gamd(x0, calib_cmd_steps, calib_rounds, calib_round_steps, dt, T,
                                     np.random.default_rng(seed + 1), sigma0=sigma0_kT * T,
                                     threshold=gamd_threshold, stride=stride, **system)
print(f"{'stage':>6} {'V_min':>7} {'V_max':>7} {'V_avg':>7} {'sigma_V':>8} {'E':>7} {'k0':>6}")
for i, h_ in enumerate(history):
    print(f"{'cMD' if i == 0 else i:>6} {h_['V_min']:7.2f} {h_['V_max']:7.2f} {h_['V_avg']:7.2f} {h_['V_std']:8.2f} "
          f"{h_['E']:7.2f} {h_['k0']:6.2f}")
stats = history[-1]
amd.check_gamd_constraints(params["E"], params["k"], stats)
print(f"\nfrozen parameters: E = {params['E']:.3f}, k = {params['k']:.4f}, k0 = {params['k0']:.3f} "
      f"({params['threshold_used']} bound)")
print(f"V_max <= E <= V_min + 1/k :  {stats['V_max']:.2f} <= {params['E']:.2f} <= {stats['V_min'] + 1 / params['k']:.2f}")
print(f"predicted sigma_dV = {params['sigma_dV_estimate'] / T:.2f} kT  (sigma0 = {sigma0_kT} kT)")

## 6. GaMD production and boost statistics

In [ ]:
t0 = timer.perf_counter()
runs["GaMD"] = amd.run_boosted_dynamics(x0, n_steps, dt, T, np.random.default_rng(seed), method="gamd",
                                         E=params["E"], k=params["k"], stride=stride, **system)
r = runs["GaMD"]
print(f"GaMD transitions = {count_transitions(r['x'], *core).sum()}, x range [{r['x'].min():+.2f}, {r['x'].max():+.2f}]"
      f"  ({timer.perf_counter() - t0:.1f} s)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(x_plot, V_slice - V_bath, color=REFERENCE, lw=1.4, label=r"$V$")
axes[0].plot(x_plot, V_slice + amd.gamd_boost(V_slice, params["E"], params["k"]) - V_bath,
             color=METHOD_COLORS["gamd"], label=r"$V^*$ GaMD")
axes[0].plot(x_plot, V_slice + amd.amd_boost(V_slice, E_amd, alpha_amd) - V_bath,
             color=METHOD_COLORS["amd"], label=r"$V^*$ aMD (moderate)")
axes[0].set_ylim(-0.8, 5.5)
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel("energy (bath at mean energy)")
axes[0].legend(fontsize=8)

for name, color in [("aMD (moderate)", METHOD_COLORS["amd"]), ("GaMD", METHOD_COLORS["gamd"])]:
    dV_s = frames(runs[name])[1] / T
    axes[1].hist(dV_s, bins=60, density=True, alpha=0.45, color=color,
                 label=f"{name}: skew {skew(dV_s):+.2f}, {np.mean(dV_s == 0):.0%} of frames at 0")
    g = np.linspace(dV_s.min(), dV_s.max(), 200)
    axes[1].plot(g, np.exp(-(g - dV_s.mean()) ** 2 / (2 * dV_s.var())) / np.sqrt(2 * np.pi * dV_s.var()),
                 color=color, lw=1.2)
    print(f"{name:15s} <dV> = {dV_s.mean():.2f} kT, std = {dV_s.std():.2f} kT, skewness = {skew(dV_s):+.2f}")
axes[1].set_ylim(0, 0.6)  # the aMD spike at dV = 0 (frames above E) is clipped
axes[1].set_xlabel(r"$\Delta V / k_BT$")
axes[1].set_ylabel("density")
axes[1].legend(fontsize=8, title="lines: Gaussian with same mean/std", title_fontsize=8)
fig.tight_layout()
save_figure(fig, "05_boost_distributions")
plt.show()

In this run the GaMD boost distribution is approximately Gaussian (small skewness) and narrow, whereas the aMD
boost is clearly skewed. That is a property of these parameters and this finite trajectory, not a guarantee.

## 7. Cumulant reweighting

For frames in bin $j$, expand $\ln\langle e^{\beta\Delta V}\rangle_j$ in cumulants and keep two terms:

$$\ln\langle e^{\beta\Delta V}\rangle_j \approx \beta\langle\Delta V\rangle_j + \frac{\beta^2}{2}\mathrm{Var}_j(\Delta V),
\qquad F(x_j) \approx -k_BT\ln p^*(x_j) - \langle\Delta V\rangle_j - \frac{\beta}{2}\mathrm{Var}_j(\Delta V).$$

This is exact only if $\Delta V$ is Gaussian within each bin; it trades the exponential's statistical noise for
a systematic approximation.

In [ ]:
x_g, dV_g = frames(runs["GaMD"])
_, F_biased, _ = histogram_free_energy(x_g, bins, T, min_count=min_count)
_, F_cum, _ = amd.cumulant_free_energy(x_g, dV_g, bins, T, order=2, min_count=min_count)
_, F_exp, _ = histogram_free_energy(x_g, bins, T, log_weights=beta * dV_g, min_count=min_count)

fig, ax = plt.subplots(figsize=(6.5, 3.6))
plot_reference(ax, xc, F_ref)
ax.plot(xc, F_biased, color=MUTED, lw=1.2, label="GaMD histogram, not reweighted")
ax.plot(xc, F_cum, color=METHOD_COLORS["gamd"], label="cumulant (2nd order)")
ax.plot(xc, F_exp, ls=":", lw=2.4, color=METHOD_COLORS["gamd"], label="exponential")
free_energy_axes(ax, ylim=(-0.2, 4.5))
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
for name, F_ in [("not reweighted", F_biased), ("cumulant", F_cum), ("exponential", F_exp)]:
    err = free_energy_rmse(F_, F_ref, region)
    print(f"{name:15s} RMSE = {err[0] / T:.2f} kT on {err[1]:.0%} of the comparison region")

## 8. aMD versus GaMD in this toy

Same number of steps for every method (GaMD additionally used a short calibration). These numbers describe one
finite run each; they are diagnostics, not a ranking. Note that cumulant reweighting of the skewed aMD boosts
is less accurate than exponential reweighting: the second-order expansion relies on near-Gaussian statistics.

In [ ]:
print(f"{'method':17s} {'trans.':>6} {'<dV>/kT':>8} {'std/kT':>7} {'skew':>6} {'ESS':>6} "
      f"{'RMSE exp':>9} {'RMSE cum':>9} {'coverage':>9}")
for name, r in runs.items():
    x_s, dV_s = frames(r)
    n_tr = count_transitions(r["x"], *core).sum()
    F_e = histogram_free_energy(x_s, bins, T, log_weights=beta * dV_s, min_count=min_count)[1]
    e_exp, cov = free_energy_rmse(F_e, F_ref, region)
    if name == "ordinary":  # no boost: plain histogram
        print(f"{name:17s} {n_tr:6d} {'-':>8} {'-':>7} {'-':>6} {'-':>6} {e_exp / T:9.2f} {'-':>9} {cov:9.0%}")
        continue
    e_cum = free_energy_rmse(amd.cumulant_free_energy(x_s, dV_s, bins, T, min_count=min_count)[1], F_ref, region)[0]
    ess = kish_effective_sample_size(beta * dV_s) / dV_s.size
    print(f"{name:17s} {n_tr:6d} {dV_s.mean() / T:8.2f} {dV_s.std() / T:7.2f} {skew(dV_s):6.2f} {ess:6.1%} "
          f"{e_exp / T:9.2f} {e_cum / T:9.2f} {cov:9.0%}")
print("RMSE in kT over the comparison region; coverage = fraction of that region with >= min_count frames.")

- **aMD**: a general boost; raising $E$ or lowering $\alpha$ gives more acceleration but wider, skewed boost
  distributions and exponential weights that can become noisy.
- **GaMD**: a harmonic boost whose parameters are constrained so that the modified potential stays monotonic and
  the boost distribution narrow, which supports approximate cumulant reweighting.

Neither is a universal winner: the useful amount of acceleration and the reliability of reweighting depend on the
system and on the parameters.

## 9. Optional experiment

Set `RUN_OPTIONAL = True` to repeat GaMD with a wider allowed boost (`sigma0 = 3 kT`). Expect more transitions,
a broader $\Delta V$ distribution and a larger difference between cumulant and exponential reweighting.

In [ ]:
RUN_OPTIONAL = False
if RUN_OPTIONAL:
    p2, _ = amd.calibrate_gamd(x0, calib_cmd_steps, calib_rounds, calib_round_steps, dt, T,
                               np.random.default_rng(seed + 1), sigma0=3.0 * T, stride=stride, **system)
    r2 = amd.run_boosted_dynamics(x0, n_steps, dt, T, np.random.default_rng(seed), method="gamd",
                                  E=p2["E"], k=p2["k"], stride=stride, **system)
    x_s, dV_s = frames(r2)
    print(f"k0 = {p2['k0']:.2f}, transitions = {count_transitions(r2['x'], *core).sum()}, "
          f"std(dV) = {dV_s.std() / T:.2f} kT, skew = {skew(dV_s):.2f}")
    for label, F_ in [("cumulant", amd.cumulant_free_energy(x_s, dV_s, bins, T, min_count=min_count)[1]),
                      ("exponential", histogram_free_energy(x_s, bins, T, log_weights=beta * dV_s, min_count=min_count)[1])]:
        print(f"  {label:12s} RMSE = {free_energy_rmse(F_, F_ref, region)[0] / T:.2f} kT")

## Takeaways

- aMD and GaMD smooth the landscape through a boost that depends only on the potential energy; the modified force is the original force times a scalar.
- Reweighting with $e^{\beta\Delta V}$ is exact in principle, but large, broad boosts make the effective sample size collapse.
- GaMD constrains $E$ and $k$ so that $V^*$ stays monotonic and $\sigma_{\Delta V}$ stays below a chosen $\sigma_0$.
- Second-order cumulant reweighting assumes near-Gaussian boost statistics within each bin; it is an approximation, not an exact identity.

*References:* Hamelberg, Mongan & McCammon, *J. Chem. Phys.* **120**, 11919 (2004), doi:10.1063/1.1755656 ·
Miao et al., *J. Chem. Theory Comput.* **10**, 2677 (2014), doi:10.1021/ct500090q ·
Miao, Feher & McCammon, *J. Chem. Theory Comput.* **11**, 3584 (2015), doi:10.1021/acs.jctc.5b00436.